<p align="center">
  <img src="archs/GraphRAG.webp" alt="GraphRAG" width="1200">
</p>

In [130]:
import os
import time
import pickle
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Tuple
from operator import itemgetter
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough

load_dotenv()

True

In [3]:
loader = PyPDFLoader("papers/Knowledge_Graphs_paper.pdf")
docs = loader.load()

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [9]:
len(chunks)

243

In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=1024,   # down from 4096 — don't need that much for constrained JSON extraction
)

In [11]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    strict_mode=True,
    allowed_nodes=["Concept", "Algorithm", "Model", "Method", "Framework", "Library", "Tool", "Database", "Dataset", "Paper", "Author", "Application"],
    allowed_relationships = ["USES", "IMPLEMENTS", "BASED_ON", "DEVELOPED_BY", "TRAINED_ON", "APPLIED_TO", "ENABLES", "DEPENDS_ON", "COMPARES_WITH", "PART_OF"],
    node_properties=False,
    relationship_properties=False
)

In [13]:
TPM_LIMIT = 6000
RPM_LIMIT = 30
TPD_LIMIT = 500000
tokens_this_minute = 0
requests_this_minute = 0
tokens_today = 0
window_start = time.time()

def convert_all(chunks, max_tokens_per_call=1024):
    global tokens_this_minute, requests_this_minute, window_start, tokens_today
    all_graph_docs = []
    failed_chunks = []
    total = len(chunks)

    for idx, chunk in enumerate(chunks):
        if time.time() - window_start > 60:
            tokens_this_minute = 0
            requests_this_minute = 0
            window_start = time.time()

        est_input_tokens = len(chunk.page_content) // 4 + 1500  # rough: schema+prompt overhead ~1500
        est_total = est_input_tokens + max_tokens_per_call

        if tokens_today + est_total > TPD_LIMIT:
            print(f"🛑 Would exceed daily token budget at chunk {idx}. Stopping — resume tomorrow.")
            failed_chunks.extend(chunks[idx:])
            break

        if tokens_this_minute + est_total > TPM_LIMIT or requests_this_minute >= RPM_LIMIT:
            wait = 60 - (time.time() - window_start) + 1
            print(f"⏳ Pacing: waiting {wait:.0f}s ({idx}/{total} done)")
            time.sleep(max(wait, 0))
            tokens_this_minute = 0
            requests_this_minute = 0
            window_start = time.time()

        try:
            graph_docs = llm_transformer.convert_to_graph_documents([chunk])
            all_graph_docs.extend(graph_docs)
            requests_this_minute += 1
            tokens_this_minute += est_total
            tokens_today += est_total
            if idx % 10 == 0:
                print(f"✅ {idx+1}/{total} — running total {len(all_graph_docs)} graph docs")
        except Exception as e:
            err = str(e)
            if "429" in err:
                print(f"⏳ Hit limit unexpectedly at chunk {idx}, backing off 30s")
                time.sleep(30)
                try:
                    graph_docs = llm_transformer.convert_to_graph_documents([chunk])
                    all_graph_docs.extend(graph_docs)
                except Exception as e2:
                    print(f"❌ Chunk {idx} failed after retry: {str(e2)[:100]}")
                    failed_chunks.append(chunk)
            else:
                print(f"❌ Chunk {idx} failed: {err[:100]}")
                failed_chunks.append(chunk)

    return all_graph_docs, failed_chunks

# Process the full 243 chunks
graph_docs, failed = convert_all(chunks)
print(f"\nTotal: {len(graph_docs)} graph docs, {len(failed)} chunks failed")

✅ 1/243 — running total 1 graph docs
⏳ Pacing: waiting 58s (2/243 done)
⏳ Pacing: waiting 59s (4/243 done)
⏳ Pacing: waiting 59s (6/243 done)
⏳ Pacing: waiting 58s (8/243 done)
⏳ Pacing: waiting 59s (10/243 done)
✅ 11/243 — running total 11 graph docs
⏳ Pacing: waiting 59s (12/243 done)
⏳ Pacing: waiting 59s (14/243 done)
⏳ Pacing: waiting 59s (16/243 done)
⏳ Pacing: waiting 58s (18/243 done)
⏳ Pacing: waiting 59s (20/243 done)
✅ 21/243 — running total 21 graph docs
⏳ Pacing: waiting 59s (22/243 done)
❌ Chunk 23 failed: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool DynamicGra
⏳ Pacing: waiting 58s (25/243 done)
⏳ Pacing: waiting 59s (27/243 done)
⏳ Pacing: waiting 59s (29/243 done)
✅ 31/243 — running total 30 graph docs
⏳ Pacing: waiting 59s (31/243 done)
❌ Chunk 32 failed: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See '
⏳ Pacing: waiting 58s (34/243 done)
⏳ Pacing: waiting 59s (36/243 done

In [4]:
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    enhanced_schema=True,
)

In [17]:
graph.add_graph_documents(
    graph_docs,
    baseEntityLabel=True,
    include_source=True
)
print(f"Ingested {len(graph_docs)} graph docs into Neo4j")

Ingested 184 graph docs into Neo4j


In [ ]:
# Saving
with open("failed_chunks.pkl", "wb") as f:
    pickle.dump(failed, f)

with open("graph_docs_batch1.pkl", "wb") as f:
    pickle.dump(graph_docs, f)

print(f"Saved {len(failed)} failed chunks and {len(graph_docs)} graph docs for later")

Saved 59 failed chunks and 184 graph docs for later


In [35]:
vector_index = Neo4jVector.from_existing_graph(
    embedding=HuggingFaceEmbeddings(model_name="thenlper/gte-small"),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

print("Created successfully")

Created successfully


c:\Users\RadhaKrishna\Documents\Lib\site-packages\neo4j\_sync\driver.py:516: ResourceWarning: unclosed  Neo4jDriver: <neo4j._sync.driver.Neo4jDriver object at 0x00000195597C29F0>.
  _unclosed_resource_warn(self)


In [6]:
driver = GraphDatabase.driver(
    uri=os.environ["NEO4J_URI"],
    auth=(
        os.environ["NEO4J_USERNAME"],
        os.environ["NEO4J_PASSWORD"]
    )
)

def showGraph(cypher):
    session = driver.session()
    widget = GraphWidget(graph=session.run(cypher).graph())
    widget.node_label_mapping = "id"
    display(widget)

In [8]:
showGraph("MATCH p=()-[:IMPLEMENTS]->() RETURN p LIMIT 30")

GraphWidget(layout=Layout(height='790px', width='100%'))

c:\Users\RadhaKrishna\Documents\Lib\site-packages\neo4j\_sync\work\workspace.py:80: ResourceWarning: unclosed  Session: <neo4j._sync.work.session.Session object at 0x00000195599D2F30>.
  _unclosed_resource_warn(self)


In [32]:
graph.query("MATCH p=()-[:BASED_ON]->() RETURN p LIMIT 25;")

[{'p': [{'id': 'Graph Databases'}, 'BASED_ON', {'id': 'Neo4J'}]},
 {'p': [{'id': 'Graph Databases'}, 'BASED_ON', {'id': 'Networkx'}]},
 {'p': [{'id': 'Graph Databases'}, 'BASED_ON', {'id': 'Tutorial'}]},
 {'p': [{'id': 'Graph Databases'}, 'BASED_ON', {'id': 'Survey'}]},
 {'p': [{'id': 'Graph Theory'},
   'BASED_ON',
   {'id': 'Social Network Analysis'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Nodes'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Edges'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Graph'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Dijkstra'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Graph Centrality'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Graph Connectivity'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Graph Database'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Neo4J'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Amazon Neptune'}]},
 {'p': [{'id': 'Paper'}, 'BASED_ON', {'id': 'Arangodb'}]},
 {'p': [{'id': 'Graph'}, 'BASED_ON

In [ ]:
graph.query("MATCH (n) RETURN labels(n) AS label, count(*) AS count ORDER BY count DESC")

[{'label': ['__Entity__', 'Concept'], 'count': 413},
 {'label': ['Document'], 'count': 184},
 {'label': ['__Entity__', 'Method'], 'count': 34},
 {'label': ['__Entity__', 'Application'], 'count': 30},
 {'label': ['__Entity__', 'Author'], 'count': 28},
 {'label': ['__Entity__', 'Algorithm'], 'count': 23},
 {'label': ['__Entity__', 'Paper'], 'count': 17},
 {'label': ['__Entity__', 'Concept', 'Method'], 'count': 12},
 {'label': ['__Entity__', 'Concept', 'Algorithm'], 'count': 9},
 {'label': ['__Entity__', 'Concept', 'Application'], 'count': 7},
 {'label': ['__Entity__', 'Library'], 'count': 7},
 {'label': ['__Entity__'], 'count': 7},
 {'label': ['__Entity__', 'Framework'], 'count': 6},
 {'label': ['__Entity__', 'Database'], 'count': 5},
 {'label': ['__Entity__', 'Model'], 'count': 4},
 {'label': ['__Entity__', 'Concept', 'Application', 'Method'], 'count': 3},
 {'label': ['__Entity__', 'Concept', 'Framework'], 'count': 3},
 {'label': ['__Entity__', 'Concept', 'Database'], 'count': 2},
 {'la

In [ ]:
graph.query("MATCH ()-[r]->() RETURN type(r) AS relation, count(*) AS count ORDER BY count DESC")

[{'relation': 'MENTIONS', 'count': 1033},
 {'relation': 'USES', 'count': 169},
 {'relation': 'APPLIED_TO', 'count': 103},
 {'relation': 'ENABLES', 'count': 67},
 {'relation': 'BASED_ON', 'count': 51},
 {'relation': 'DEVELOPED_BY', 'count': 24},
 {'relation': 'PART_OF', 'count': 20},
 {'relation': 'IMPLEMENTS', 'count': 18},
 {'relation': 'DEPENDS_ON', 'count': 14},
 {'relation': 'COMPARES_WITH', 'count': 10},
 {'relation': 'TRAINED_ON', 'count': 4}]

In [ ]:
graph.query("MATCH (a:Algorithm) RETURN a.id LIMIT 10")

[{'a.id': 'Dijkstra'},
 {'a.id': 'Louvain Method'},
 {'a.id': 'Graph Neural Networks'},
 {'a.id': 'Dijkstra’S'},
 {'a.id': 'Bellman-Ford'},
 {'a.id': 'Link Prediction Algorithms'},
 {'a.id': 'Gnns'},
 {'a.id': 'Cypher'},
 {'a.id': 'Degree Centrality'},
 {'a.id': "Dijkstra'S Algorithm"}]

In [29]:
graph.query("""MATCH (n) WHERE toLower(n.id) CONTAINS "neo4j" RETURN labels(n), n.id""")

[{'labels(n)': ['__Entity__',
   'Concept',
   'Tool',
   'Database',
   'Library',
   'Framework'],
  'n.id': 'Neo4J'},
 {'labels(n)': ['__Entity__', 'Tool'], 'n.id': 'Neo4J Bloom'},
 {'labels(n)': ['__Entity__', 'Library'], 'n.id': 'Neo4J Python Driver'},
 {'labels(n)': ['__Entity__', 'Application'], 'n.id': 'Neo4J Desktop'},
 {'labels(n)': ['__Entity__', 'Database'], 'n.id': 'Neo4J Databases'}]

In [9]:
class Entities(BaseModel):
    """Identifying information about entities."""
    names: List[str] = Field(
        ...,
        description="All the concepts, tools, frameworks, methods, papers, or authors that appear in the text"
    )

In [12]:
entity_prompt = ChatPromptTemplate.from_messages([
    ("system",
        """Extract only the domain-specific entities that are useful for graph retrieval.
        Extract:
        - Concepts
        - Algorithms
        - Methods
        - Frameworks
        - Tools
        - Libraries
        - Models
        - Databases
        - Papers
        - Authors

        Ignore:
        - Pronouns
        - Generic words
        - Verbs
        - Numbers
        - Citations
        - Stopwords

        Return only entities that appear explicitly in the input.
        """),
    ("human", "{question}")])

entity_chain = entity_prompt | llm.with_structured_output(Entities)

In [13]:
entity_chain.invoke("What is Neo4j and how is it related to Dijkstra's Algorithm?")

Entities(names=['Neo4j', "Dijkstra's Algorithm"])

In [14]:
# --- Structured (graph-based) retriever ---
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    full_text_query = "".join(f" {w}~2 AND" for w in words[:-1])
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

In [15]:
generate_full_text_query("What is Neo4j and how is it related to Dijkstra's Algorithm?")

"What~2 AND is~2 AND Neo4j~2 AND and~2 AND how~2 AND is~2 AND it~2 AND related~2 AND to~2 AND Dijkstra's~2 AND Algorithm~2"

In [16]:
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:5})
            YIELD node, score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join(el['output'] for el in response)
    return result

In [17]:
print(structured_retriever("What is Neo4j and how is it related to Dijkstra's Algorithm?"))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:5})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

Neo4J - IMPLEMENTS -> Cypher Query Language
Neo4J - IMPLEMENTS -> Database
Neo4J - IMPLEMENTS -> Subgraph
Neo4J - IMPLEMENTS -> Core Resilience
Neo4J - IMPLEMENTS -> Decision Tree Model
Neo4J - USES -> Graph Databases
Neo4J - USES -> Nodes
Neo4J - USES -> Edges
Neo4J - USES -> Graph
Neo4J - USES -> Gnns
Neo4J - USES -> Entities
Neo4J - USES -> Graph Structures
Neo4J - USES -> Apoc
Neo4J - USES -> Neo4J Python Driver
Neo4J - USES -> Localdbms
Neo4J - USES -> Contextual Data
Neo4J - ENABLES -> Acid Transactions
Neo4J - ENABLES -> Transactional Operations
Neo4J - ENABLES -> Analytical Operations
Neo4J - ENABLES -> Influential Nodes
Neo4J - ENABLES -> Resilience Studies
Neo4J - ENABLES -> Influencer Detection
Neo4J - ENABLES -> Community Clustering
Neo4J - ENABLES -> Recommendation Engine
Neo4J - ENABLES -> Recommender System
Neo4J - ENABLES -> Social Network Analysis Tool
Neo4J - APPLIED_TO -> Social Networks
Neo4J - APPLIED_TO -> Recommendation Systems
Neo4J - APPLIED_TO -> Fraud Detecti

In [43]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    docs = vector_index.similarity_search(question, k=5)
    unstructured_data = []

    for i, doc in enumerate(docs, start=1):
        unstructured_data.append(f"Document {i}:\n{doc.page_content}")

    return f"""
        Structured data:
        {structured_data}

        Unstructured data:
        {chr(10).join(unstructured_data)}
    """

In [45]:
print(retriever("What is Neo4j and how is it related to Dijkstra's Algorithm?"))

Search query: What is Neo4j and how is it related to Dijkstra's Algorithm?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:5})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            


        Structured data:
        Neo4J - IMPLEMENTS -> Cypher Query Language
Neo4J - IMPLEMENTS -> Database
Neo4J - IMPLEMENTS -> Subgraph
Neo4J - IMPLEMENTS -> Core Resilience
Neo4J - IMPLEMENTS -> Decision Tree Model
Neo4J - USES -> Graph Databases
Neo4J - USES -> Nodes
Neo4J - USES -> Edges
Neo4J - USES -> Graph
Neo4J - USES -> Gnns
Neo4J - USES -> Entities
Neo4J - USES -> Graph Structures
Neo4J - USES -> Apoc
Neo4J - USES -> Neo4J Python Driver
Neo4J - USES -> Localdbms
Neo4J - USES -> Contextual Data
Neo4J - ENABLES -> Acid Transactions
Neo4J - ENABLES -> Transactional Operations
Neo4J - ENABLES -> Analytical Operations
Neo4J - ENABLES -> Influential Nodes
Neo4J - ENABLES -> Resilience Studies
Neo4J - ENABLES -> Influencer Detection
Neo4J - ENABLES -> Community Clustering
Neo4J - ENABLES -> Recommendation Engine
Neo4J - ENABLES -> Recommender System
Neo4J - ENABLES -> Social Network Analysis Tool
Neo4J - APPLIED_TO -> Social Networks
Neo4J - APPLIED_TO -> Recommendation Systems
N

In [ ]:
condense_llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)

In [146]:
CONDENSE_QUESTION_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", 
        """
        Given the conversation history and a follow-up question, rewrite the follow-up
        question so that it is a complete standalone question.

        Requirements:
        - Preserve the original meaning.
        - Preserve all technical terms, entity names, and relationships.
        - Resolve pronouns using the chat history.
        - Do not answer the question.
        - Return only the rewritten standalone question.
        """
        ),
        MessagesPlaceholder(variable_name="chat_history", n_messages=4),  # last 2 turns only
        ("human", "Follow-up Question: {question}"),
    ]
)

In [153]:
search_query = RunnableBranch(

    (
        RunnableLambda(lambda x: "chat_history" in x and len(x["chat_history"]) > 0),

        RunnablePassthrough.assign(
            chat_history=lambda x: x.get("chat_history", [])
        )
        | CONDENSE_QUESTION_PROMPT
        | condense_llm
        | StrOutputParser(),
    ),

    RunnableLambda(lambda x: x["question"]),
)

In [134]:
# Just testin' around
search_query.invoke({
    "question": "How does it work?",
    "chat_history": [
        HumanMessage(content="What is Neo4j?"),
        AIMessage(content="Neo4j is a graph database."),
    ]
})

'What is the underlying architecture of Neo4j that enables it to efficiently store and query graph data?'

In [157]:
chat_history = []

def make_chat_history(x):
    if x["answer"].strip() != "NO_ANSWER_FOUND":
        chat_history.append(HumanMessage(content=x["question"]))
        chat_history.append(AIMessage(content=x["answer"]))
        return x["answer"]
    else:
        return "That information isn't available in the documents I have."

In [155]:
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful AI assistant.

        Answer ONLY using the provided context.

        If the answer cannot be found in the context, respond with exactly:
        NO_ANSWER_FOUND

        Do not invent facts. Be concise and accurate.
        """
    ),
    (
        "human",
        """Context:
        {context}

        Question:
        {question}
        """
    )
])

In [158]:
chain = (
    RunnableParallel({
        "context": search_query | retriever,
        "question": itemgetter("question")
    })
    | RunnablePassthrough.assign(
        answer=answer_prompt | llm | StrOutputParser()
    )
    | RunnableLambda(make_chat_history)
)

In [160]:
print(chain.invoke({
    "question": "What is Neo4j?",
    "chat_history": chat_history
}))

Search query: What is Neo4j?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:5})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

Neo4j is a database that IMPLEMENTS a Database, Subgraph, Core Resilience, and Decision Tree Model, and it USES Graph Databases, Nodes, Edges, Graph, Gnns, Entities, Graph Structures, Apoc, Neo4J Python Driver, Localdbms, and Contextual Data.


In [139]:
print(chain.invoke({
    "question": "Who is Shinjan?",
    "chat_history": chat_history
}))

Search query: What is the relationship between Shinjan and Neo4j, considering the context of the conversation about Neo4j?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:5})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

Information is not available.
